In [177]:
import pandas as pd
import numpy as np

df = pd.read_csv('../Our Datasets/features_sample_match_new.csv')

start_types_wants = ['pass_interception', 'recovery']

df = df[df['start_type'].isin(start_types_wants)]

df['is_drawing'] = np.where(df['game_state'] == 'drawing', 1, 0)

df['is_middle_third'] = np.where(df['third_start'] == 'middle', 1, 0)

# dropping irrelevant columns
cols_to_drop = ['Unique ID', 'match_id', 'event_index', 'frame_anchor', 'rec_player_id', 'rec_team_short', 'start_type', 'end_type', 'event_row_index', 
                'source_file', 'error', 'dist_to_near_goal', 'dist_to_far_goal', 'third_end', 'game_state', 'n_passing_options_dangerous_not_difficult', 'third_start',
                'n_forward_options', 'd_nearest_opp', 'd_nearest_team', 'mean_team_dist', 'n_opp_within5']

df = df.drop(columns=cols_to_drop)

df['team_out_of_possession_phase_type'] = (
    df['team_out_of_possession_phase_type']
        .replace({
            'defending_set_play': 'set_play',
            'defending_quick_break': 'transition/quick_break',
            'defending_direct': 'direct',
            'defending_transition': 'transition/quick_break',
            'chaotic': 'chaotic/disruption',
            'disruption': 'chaotic/disruption'
        })
)


#df['team_out_of_possession_phase_type'] = df['team_out_of_possession_phase_type'].str.replace('high_block', 'block')
#df['team_out_of_possession_phase_type'] = df['team_out_of_possession_phase_type'].str.replace('medium_block', 'block')
#df['team_out_of_possession_phase_type'] = df['team_out_of_possession_phase_type'].str.replace('low_block', 'block')

In [178]:
df

,dist_to_attacking_goal,team_out_of_possession_phase_type,max_player_targeted_xthreat,player_targeted_dangerous,is_drawing,is_middle_third
0,99.575170,high_block,0.0024,0.0,1,0
3,34.427403,chaotic/disruption,0.0091,0.0,1,0
9,82.713146,chaotic/disruption,0.0091,0.0,1,0
14,51.188596,chaotic/disruption,0.0359,1.0,1,1
19,47.054798,chaotic/disruption,0.0359,1.0,1,1
...,...,...,...,...,...,...
5850,82.279522,chaotic/disruption,NaN,NaN,0,0
5851,52.203530,chaotic/disruption,NaN,NaN,0,1
5852,34.236218,chaotic/disruption,0.0133,0.0,0,0
5853,87.764356,chaotic/disruption,NaN,NaN,0,0


In [179]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor  # pip install xgboost if needed

In [180]:
#third_order = ["defensive", "middle", "attacking"]
#df["third_start"] = df["third_start"].map({v: i for i, v in enumerate(third_order)})

#df = df[['dist_to_attacking_goal', 'max_player_targeted_xthreat', 'team_out_of_possession_phase_type']]

game_state_map = {
    "losing": -1,
    "drawing": 0,
    "winning": 1
}
#df["game_state"] = df["game_state"].map(game_state_map)

df = pd.get_dummies(df, columns=["team_out_of_possession_phase_type"], prefix="oop_phase")

del df['player_targeted_dangerous']

target_col = "max_player_targeted_xthreat"
df = df[df[target_col].notna()]

In [181]:
df

,dist_to_attacking_goal,max_player_targeted_xthreat,is_drawing,is_middle_third,oop_phase_chaotic/disruption,oop_phase_direct,oop_phase_high_block,oop_phase_low_block,oop_phase_medium_block,oop_phase_set_play,oop_phase_transition/quick_break
0,99.575170,0.0024,1,0,False,False,True,False,False,False,False
3,34.427403,0.0091,1,0,True,False,False,False,False,False,False
9,82.713146,0.0091,1,0,True,False,False,False,False,False,False
14,51.188596,0.0359,1,1,True,False,False,False,False,False,False
19,47.054798,0.0359,1,1,True,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...
5844,84.239471,0.0006,0,0,False,True,False,False,False,False,False
5845,71.880025,0.0064,0,0,True,False,False,False,False,False,False
5847,60.018211,0.0014,0,1,True,False,False,False,False,False,False
5848,78.036013,0.0026,0,0,True,False,False,False,False,False,False


In [182]:
df_model = df.copy()

target_col = "max_player_targeted_xthreat"
df_model = df_model[df_model[target_col].notna()]

X = df_model.drop(columns=["max_player_targeted_xthreat"])
y = df_model["max_player_targeted_xthreat"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42
)

In [183]:
rf = RandomForestRegressor(
    n_estimators=75,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

RandomForestRegressor(n_estimators=75, n_jobs=-1, random_state=42)

In [184]:
xgb = XGBRegressor(
    n_estimators=50,
    random_state=42,
    n_jobs=-1,
    objective="reg:squarederror"
)
xgb.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=50,
             n_jobs=-1, num_parallel_tree=None, ...)

In [185]:
def evaluate(model, X_train, y_train, X_test, y_test, name):
    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)

    print(f"\n📌 {name} Performance")
    print(f"Train RMSE: {np.sqrt(mean_squared_error(y_train, pred_train))} | "
          f"R2: {r2_score(y_train, pred_train):.3f}")
    print(f"Test  RMSE: {np.sqrt(mean_squared_error(y_test, pred_test))} | "
          f"R2: {r2_score(y_test, pred_test):.3f}")

evaluate(rf, X_train, y_train, X_test, y_test, "Random Forest")
evaluate(xgb, X_train, y_train, X_test, y_test, "XGBoost")


📌 Random Forest Performance
Train RMSE: 0.014039330304788671 | R2: 0.858
Test  RMSE: 0.048054622493632675 | R2: 0.171

📌 XGBoost Performance
Train RMSE: 0.015060142138516005 | R2: 0.837
Test  RMSE: 0.048994539705687686 | R2: 0.138


In [186]:
y_mean = np.full_like(y_test, y_train.mean(), dtype=float)

baseline_rmse = np.sqrt(mean_squared_error(y_test, y_mean))
baseline_r2   = r2_score(y_test, y_mean)

print("Baseline RMSE:", baseline_rmse)
print("Baseline R2:", baseline_r2)

Baseline RMSE: 0.052834685465648804
Baseline R2: -0.00221018682994667


In [187]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=0.1)
ridge.fit(X_train, y_train)
evaluate(ridge, X_train, y_train, X_test, y_test, "Ridge")


📌 Ridge Performance
Train RMSE: 0.03513215059961925 | R2: 0.112
Test  RMSE: 0.0508555829836843 | R2: 0.071


In [188]:
from sklearn.linear_model import LinearRegression

linreg = LinearRegression()
linreg.fit(X_train, y_train)

evaluate(linreg, X_train, y_train, X_test, y_test, "Linear Regression")


📌 Linear Regression Performance
Train RMSE: 0.035132147250668835 | R2: 0.112
Test  RMSE: 0.0508571191930885 | R2: 0.071
